# Part 7 — Multidimensional analysis on the Silver layer

Three things in this notebook:
1. **Business questions** answered with plain SQL on `silver_ticks`:
   - average price per symbol per day,
   - which symbol is most volatile,
   - peak trading hours.
2. **OLAP operators** — GROUP BY, ROLLUP, CUBE — on the same data.
3. **Star-schema design** — we materialise `fact_ticks`, `dim_time`, `dim_symbol`, `dim_market` as Parquet under `/lab3/gold_star/` and register them as Spark SQL tables so Superset can query them too.

**Prerequisite.** Notebook `04_hdfs_medallion.ipynb` must have run long enough to produce `/lab3/silver/ticks` Parquet files.

## 1. Spark session + load Silver

In [ ]:
import findspark; findspark.init()
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("MultidimAnalysis")
    .master("local[*]")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000")
    .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

silver = spark.read.parquet("hdfs://namenode:9000/lab3/silver/ticks")
silver.createOrReplaceTempView("silver_ticks")
print("silver rows:", silver.count())
silver.printSchema()

## 2. Business questions

### Q1 — Average price per symbol per day

In [ ]:
spark.sql("""
  SELECT ingest_date,
         symbol,
         ROUND(AVG(price), 6) AS avg_price,
         COUNT(*)             AS n_ticks
  FROM silver_ticks
  GROUP BY ingest_date, symbol
  ORDER BY ingest_date, symbol
""").show(50, truncate=False)

### Q2 — Which symbol is most volatile?

Volatility is measured as the standard deviation of price within the period, normalised by the mean — i.e. the *coefficient of variation*. This makes the metric comparable across symbols whose absolute price differs by orders of magnitude (BTC at $77000 vs ADA at $0.25).

In [ ]:
spark.sql("""
  SELECT symbol,
         ROUND(AVG(price), 6)                    AS mean_price,
         ROUND(STDDEV_SAMP(price), 6)            AS stddev_price,
         ROUND(STDDEV_SAMP(price) / AVG(price)
               * 100, 4)                          AS cv_pct,
         COUNT(*)                                AS n_ticks
  FROM silver_ticks
  GROUP BY symbol
  ORDER BY cv_pct DESC
""").show(truncate=False)

### Q3 — Peak trading hours

Group ticks by hour of day (UTC) and rank by total volume.

In [ ]:
spark.sql("""
  SELECT HOUR(event_time)             AS hour_utc,
         COUNT(*)                     AS n_ticks,
         ROUND(SUM(volume), 2)        AS total_volume
  FROM silver_ticks
  GROUP BY HOUR(event_time)
  ORDER BY total_volume DESC
""").show(24, truncate=False)

## 3. ROLLUP and CUBE

Both produce **subtotal rows** at multiple aggregation levels in a single query.

- `ROLLUP(a, b, c)` → 4 grouping levels: `(a,b,c)`, `(a,b)`, `(a)`, `()` — a hierarchical drill-up.
- `CUBE(a, b)` → 4 grouping levels: `(a,b)`, `(a)`, `(b)`, `()` — every combination.

### ROLLUP — total volume by (date, symbol) with subtotals

In [ ]:
spark.sql("""
  SELECT ingest_date,
         symbol,
         ROUND(SUM(volume), 2) AS total_volume,
         COUNT(*)              AS n_ticks
  FROM silver_ticks
  GROUP BY ROLLUP(ingest_date, symbol)
  ORDER BY ingest_date NULLS LAST, symbol NULLS LAST
""").show(50, truncate=False)

Rows where one of `ingest_date` / `symbol` is `null` are the **subtotal** rows produced by ROLLUP (per date all symbols, then grand total).

### CUBE — same but every combination including "by symbol across all dates"

In [ ]:
spark.sql("""
  SELECT ingest_date,
         symbol,
         ROUND(SUM(volume), 2) AS total_volume
  FROM silver_ticks
  GROUP BY CUBE(ingest_date, symbol)
  ORDER BY ingest_date NULLS LAST, symbol NULLS LAST
""").show(50, truncate=False)

## 4. Star-schema design

We split `silver_ticks` into a fact table (the measurements) and three dimension tables (the descriptive context):

```
                       +------------+
                       | dim_time   |
                       +------------+
                              ^
                              | time_key
                              |
          +------------+      |       +------------+
          | dim_symbol |<----fact_ticks----->| dim_market |
          +------------+      |       +------------+
                              | (no other measures here
                              |  — price + volume only)
                              +------------+
```

**Why split it?** The fact table is *narrow* and *long* (millions of rows over time), while the dimensions are *wide* and *short* (a few rows describing each symbol or market). Joining on integer keys is fast; storing the descriptive strings denormalised in the fact table would multiply storage cost.

We materialise these as Parquet under `/lab3/gold_star/` and register them as Spark tables.

In [ ]:
from pyspark.sql.functions import (
    col, year, month, dayofmonth, hour, minute, dayofweek, lit,
    monotonically_increasing_id
)
from pyspark.sql.types import IntegerType

GOLD = "hdfs://namenode:9000/lab3/gold_star"

# --- dim_time : one row per distinct event_time (rounded to the second) ---
dim_time = (silver.select(col("event_time").alias("time_key"))
    .distinct()
    .withColumn("year",       year(col("time_key")))
    .withColumn("month",      month(col("time_key")))
    .withColumn("day",        dayofmonth(col("time_key")))
    .withColumn("hour",       hour(col("time_key")))
    .withColumn("minute",     minute(col("time_key")))
    .withColumn("weekday",    dayofweek(col("time_key"))))

dim_time.write.mode("overwrite").parquet(f"{GOLD}/dim_time")
print("dim_time rows:", dim_time.count())
dim_time.show(5, truncate=False)

In [ ]:
# --- dim_symbol : one row per traded symbol -----------------------------
# Symbols on Binance follow the pattern <BASE><QUOTE>, e.g. BTCUSDT = BTC/USDT.
from pyspark.sql.functions import expr

dim_symbol = (silver.select("symbol").distinct()
    .withColumn("base_asset",  expr("substring(symbol, 1, length(symbol) - 4)"))
    .withColumn("quote_asset", expr("substring(symbol, length(symbol) - 3, 4)"))
    .withColumnRenamed("symbol", "symbol_key"))

dim_symbol.write.mode("overwrite").parquet(f"{GOLD}/dim_symbol")
print("dim_symbol rows:", dim_symbol.count())
dim_symbol.show(truncate=False)

In [ ]:
# --- dim_market : one row, since all our data is from Binance Spot ----
dim_market = spark.createDataFrame(
    [("BINANCE_SPOT", "Binance", "SPOT", "USD")],
    ["market_key", "exchange", "venue_type", "settlement_currency"]
)
dim_market.write.mode("overwrite").parquet(f"{GOLD}/dim_market")
dim_market.show()

In [ ]:
# --- fact_ticks : narrow table linking to all dimensions via foreign keys --
fact_ticks = (silver
    .select(
        col("event_time").alias("time_key"),     # FK -> dim_time
        col("symbol").alias("symbol_key"),       # FK -> dim_symbol
        lit("BINANCE_SPOT").alias("market_key"), # FK -> dim_market
        col("price"),
        col("volume"),
        col("ingest_date"),
    ))

(fact_ticks.write.mode("overwrite")
    .partitionBy("ingest_date")
    .parquet(f"{GOLD}/fact_ticks"))
print("fact_ticks rows:", fact_ticks.count())
fact_ticks.show(5, truncate=False)

### Sample star query — total volume per base asset, per day

Demonstrates the schema's purpose: the fact table holds the measures, the dimensions add context (here `base_asset` from `dim_symbol`).

In [ ]:
fact = spark.read.parquet(f"{GOLD}/fact_ticks")
dim_s = spark.read.parquet(f"{GOLD}/dim_symbol")
fact.createOrReplaceTempView("fact_ticks")
dim_s.createOrReplaceTempView("dim_symbol")

spark.sql("""
  SELECT f.ingest_date,
         s.base_asset,
         ROUND(SUM(f.volume), 2) AS total_volume,
         COUNT(*)                AS n_ticks
  FROM fact_ticks f
  JOIN dim_symbol s
    ON f.symbol_key = s.symbol_key
  GROUP BY f.ingest_date, s.base_asset
  ORDER BY f.ingest_date, total_volume DESC
""").show(truncate=False)

## 5. Cleanup

After this notebook runs once, the star-schema tables also live in HDFS at `/lab3/gold_star/`. Re-running the table-registration script will pick them up so Superset can use them too:

```bash
docker exec lab3-spark-thrift /opt/spark/bin/beeline \
  -u "jdbc:hive2://localhost:10000" -n spark \
  -e "CREATE TABLE IF NOT EXISTS fact_ticks USING parquet LOCATION 'hdfs://namenode:9000/lab3/gold_star/fact_ticks';
      CREATE TABLE IF NOT EXISTS dim_time   USING parquet LOCATION 'hdfs://namenode:9000/lab3/gold_star/dim_time';
      CREATE TABLE IF NOT EXISTS dim_symbol USING parquet LOCATION 'hdfs://namenode:9000/lab3/gold_star/dim_symbol';
      CREATE TABLE IF NOT EXISTS dim_market USING parquet LOCATION 'hdfs://namenode:9000/lab3/gold_star/dim_market';"
```

(These four lines were also added to `scripts/setup_thrift_tables.sql` so they get registered automatically next time the script runs.)

In [ ]:
spark.stop()